In [1]:
from datasets import load_dataset

corpus  = load_dataset("BeIR/fiqa", "corpus")["corpus"]     # the haystack (~57k docs)
queries = load_dataset("BeIR/fiqa", "queries")["queries"]   # the questions
qrels   = load_dataset("BeIR/fiqa-qrels") 

c:\Users\nsuh\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(corpus)

Dataset({
    features: ['_id', 'title', 'text'],
    num_rows: 57638
})


In [3]:
# datasets are row-indexed; we need id-indexed, so build lookups once
query_text = {row["_id"]: row["text"] for row in queries}
doc_text   = {row["_id"]: row["text"] for row in corpus}

one = qrels['train'][0]
qid = str(one["query-id"])
did = str(one["corpus-id"])


In [4]:
print("QUERY:", query_text[qid])
print("DOC  :", doc_text[did])

QUERY: What is considered a business expense on a business trip?
DOC  : The IRS Guidance pertaining to the subject.  In general the best I can say is your business expense may be deductible.  But it depends on the circumstances and what it is you want to deduct. Travel Taxpayers who travel away from home on business may deduct related   expenses, including the cost of reaching their destination, the cost   of lodging and meals and other ordinary and necessary expenses.   Taxpayers are considered “traveling away from home” if their duties   require them to be away from home substantially longer than an   ordinary day’s work and they need to sleep or rest to meet the demands   of their work. The actual cost of meals and incidental expenses may be   deducted or the taxpayer may use a standard meal allowance and reduced   record keeping requirements. Regardless of the method used, meal   deductions are generally limited to 50 percent as stated earlier.    Only actual costs for lodging may 

In [5]:
from collections import defaultdict

train_pairs = []
for row in qrels['train']:
    qid, did = str(row['query-id']), str(row['corpus-id'])
    if row["score"] > 0:
        train_pairs.append((query_text[qid], doc_text[did]))

relevant = defaultdict(set)
for row in qrels["test"]:
    if row["score"] > 0:
        relevant[str(row["query-id"])].add(str(row["corpus-id"]))

print("train pairs:", len(train_pairs))
print("test queries with answers:", len(relevant))
print("example pair:", train_pairs[3])

train pairs: 14166
test queries with answers: 648
example pair: ('“Business day” and “due date” for bills', "I don't believe Saturday is a business day either. When I deposit a check at a bank's drive-in after 4pm Friday, the receipt tells me it will credit as if I deposited on Monday.  If a business' computer doesn't adjust their billing to have a weekday due date, they are supposed to accept the payment on the next business day, else, as you discovered, a Sunday due date is really the prior Friday. In which case they may be running afoul of the rules that require X number of days from the time they mail a bill to the time it's due.  The flip side to all of this, is to pick and choose your battles in life. Just pay the bill 2 days early. The interest on a few hundred dollars is a few cents per week. You save that by not using a stamp, just charge it on their site on the Friday. Keep in mind, you can be right, but their computer still dings you. So you call and spend your valuable time

In [8]:
import torch
from transformers import AutoModel, AutoTokenizer


In [10]:
tokenizer = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7471.95it/s]


In [11]:
from model import Encoder
enc = Encoder()
v = enc.encode(["what is a 401k?", "how do dividends work?"])
print(v.shape)        # expect (2, 384) for MiniLM
print(v.norm(dim=1))  # expect ~[1.0, 1.0] — confirms normalization worked

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6458.76it/s]

torch.Size([2, 384])
tensor([1.0000, 1.0000])
